In [ ]:
# Import & Load data (pastikan paket penting terpasang cek requirements.txt)
import pandas as pd
import numpy as np
from pathlib import Path

# load data master
data_candidates = [
    Path('data') / 'master' / 'Master Data.xlsx',
    Path('..') / 'data' / 'master' / 'Master Data.xlsx',
    Path.cwd().parent / 'data' / 'master' / 'Master Data.xlsx',
]
path = next((p for p in data_candidates if p.exists()), data_candidates[0])
print('Using data file:', path.resolve())
df = pd.read_excel(path)

print(df.shape)
df.head()

# Hitung penduduk + fitur rasio & densitas
df["penduduk_2024"] = df["kepadatan_penduduk"] * df["luas_km2"]

layanan_cols = ["apotek", "puskesmas_total", "klinik", "rsu_total"]

for c in layanan_cols:
    df[f"rasio_{c}"] = df[c] / df["penduduk_2024"] * 10000 #rumus hitung rasio ketersediaan layanan 
    df[f"dens_{c}"]  = df[c] / df["luas_km2"]  # rumus hitung kepadatan Apotek

# Buat skor layanan + label kelas (Rendah/Sedang/Tinggi) 

rasio_cols = [f"rasio_{c}" for c in layanan_cols]

z = (df[rasio_cols] - df[rasio_cols].mean()) / df[rasio_cols].std(ddof=0)
df["skor_layanan"] = z.mean(axis=1)

q1, q2 = df["skor_layanan"].quantile([1/3, 2/3])

df["kelas_layanan"] = pd.cut(
    df["skor_layanan"],
    bins=[-np.inf, q1, q2, np.inf],
    labels=["Rendah", "Sedang", "Tinggi"],
)

print(df["kelas_layanan"].value_counts())

# menyimpan hasil fitur tambahan dan label kelas ke file excel baru
out_candidates = [
    Path('data') / 'master' / 'Master Data - fitur & label.xlsx',
    Path('outputs') / 'Master Data - fitur & label.xlsx',
    Path('..') / 'data' / 'master' / 'Master Data - fitur & label.xlsx',
]
out_path = next((p for p in out_candidates if p.parent.exists()), out_candidates[0])
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_excel(out_path, index=False)
print('Saved to:', out_path.resolve())

Using data file: C:\Users\femmy\OneDrive\Documents\6. Semester 6\tugas_akhir_machine_learning-main\data\master\Master Data.xlsx
(27, 8)
kelas_layanan
Rendah    9
Sedang    9
Tinggi    9
Name: count, dtype: int64
Saved to: C:\Users\femmy\OneDrive\Documents\6. Semester 6\tugas_akhir_machine_learning-main\notebooks\outputs\Master Data - fitur & label.xlsx
